# Multilevel Modeling and Time-Series Forecasting

In this lab you will fit multilevel (mixed-effects) models to nested behavioral data and compare them to a single-level regression that ignores the nesting structure. The dataset contains session-level response rates from eight participants, each observed across twenty sessions under varying reinforcement rates.

**Learning objectives:**
- Understand why ignoring nesting leads to biased inference
- Fit and interpret mixed-effects models using `statsmodels`
- Compare models using AIC and BIC
- Visualize participant-level variation in intercepts and slopes

**Required packages:** `pandas`, `numpy`, `matplotlib`, `statsmodels`, `scipy`

## Setup

Run the cell below to import the libraries you will need.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)

## Task 1: Load and Explore the Data

Load `nested_behavior_data.csv` into a pandas DataFrame. The dataset has four columns:

| Column | Description |
|---|---|
| `participant_id` | Participant identifier (1-8) |
| `session` | Session number (1-20) |
| `reinforcement_rate` | Reinforcements delivered per minute |
| `response_rate` | Responses per minute |

After loading:
1. Print the first few rows and the shape of the DataFrame.
2. Compute summary statistics grouped by participant.
3. Create a scatter plot of `reinforcement_rate` vs. `response_rate`, coloring points by participant.

In [ ]:
df = pd.read_csv('nested_behavior_data.csv')
print(df.shape)
df.head()

In [ ]:
# Summary statistics by participant
summary = df.groupby('participant_id')[['reinforcement_rate', 'response_rate']].agg(['mean', 'std', 'min', 'max'])
print(summary)

In [ ]:
# Scatter plot colored by participant
fig, ax = plt.subplots()
for pid, g in df.groupby('participant_id'):
    ax.scatter(g['reinforcement_rate'], g['response_rate'], label=f'P{pid}', alpha=0.7)
ax.set_xlabel('Reinforcement rate (per min)')
ax.set_ylabel('Response rate (per min)')
ax.set_title('Response vs. reinforcement rate, colored by participant')
ax.legend(title='Participant', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Task 2: Fit a Single-Level OLS Regression

Fit an ordinary least squares (OLS) regression predicting `response_rate` from `reinforcement_rate`, pooling all participants together. This model ignores the nesting of observations within participants.

1. Fit the model using `smf.ols()`.
2. Print the summary.
3. Plot the residuals vs. fitted values. Do you see any patterns that suggest the model is misspecified?
4. Color the residual plot by participant. What do you notice?

In [ ]:
ols_result = smf.ols('response_rate ~ reinforcement_rate', data=df).fit()
print(ols_result.summary())

In [ ]:
# Residual plot colored by participant
df['ols_fitted'] = ols_result.fittedvalues
df['ols_resid'] = ols_result.resid

fig, ax = plt.subplots()
for pid, g in df.groupby('participant_id'):
    ax.scatter(g['ols_fitted'], g['ols_resid'], label=f'P{pid}', alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Fitted values')
ax.set_ylabel('Residuals')
ax.set_title('OLS residuals vs. fitted, colored by participant')
ax.legend(title='Participant', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Residuals cluster by participant -> observations within a participant are correlated.
print('Mean residual by participant:')
print(df.groupby('participant_id')['ols_resid'].mean())

**Question:** Why might the standard errors from the OLS model be misleading when the data are nested? Write your answer below.

Observations within a participant are not independent -- each participant's residuals share a common offset (the per-participant mean residuals above are far from zero, ranging from about -8 to +10). OLS assumes independent errors, so it treats 160 correlated observations as if they were 160 independent ones. This overstates the effective sample size, shrinks the standard errors, and inflates the t-statistics, making effects look more precise and more significant than they really are.

## Task 3: Fit a Random-Intercept Model

Now fit a multilevel model that allows each participant to have their own intercept, while the slope of `reinforcement_rate` is fixed across participants. Use `smf.mixedlm()` from statsmodels.

```python
model = smf.mixedlm("response_rate ~ reinforcement_rate",
                     data=df,
                     groups=df["participant_id"])
result = model.fit()
```

1. Fit the model and print the summary.
2. What is the estimated variance of the random intercept?
3. How does the fixed-effect slope compare to the OLS slope from Task 2?

In [ ]:
# Fit with maximum likelihood (reml=False) so AIC/BIC are comparable across models
ri_model = smf.mixedlm('response_rate ~ reinforcement_rate', data=df, groups=df['participant_id'])
ri_result = ri_model.fit(reml=False)
print(ri_result.summary())

print('\nRandom-intercept variance (Group Var):', round(ri_result.cov_re.iloc[0, 0], 3))
print('Mixed-model slope:', round(ri_result.fe_params['reinforcement_rate'], 3),
      '| OLS slope:', round(ols_result.params['reinforcement_rate'], 3))

## Task 4: Fit a Random-Intercept-and-Slope Model

Extend the model so that each participant has both a unique intercept and a unique slope for `reinforcement_rate`. In statsmodels, you specify the random slope using the `re_formula` argument:

```python
model_rs = smf.mixedlm("response_rate ~ reinforcement_rate",
                        data=df,
                        groups=df["participant_id"],
                        re_formula="~reinforcement_rate")
result_rs = model_rs.fit()
```

1. Fit the model and print the summary.
2. Extract the random effects for each participant using `result_rs.random_effects`.
3. For each participant, compute the participant-specific intercept and slope (fixed effect + random effect).

In [ ]:
rs_model = smf.mixedlm('response_rate ~ reinforcement_rate', data=df,
                       groups=df['participant_id'], re_formula='~reinforcement_rate')
rs_result = rs_model.fit(reml=False)
print(rs_result.summary())

In [ ]:
# Participant-specific intercepts and slopes = fixed effect + random effect
fe_int = rs_result.fe_params['Intercept']
fe_slope = rs_result.fe_params['reinforcement_rate']

rows = []
for pid, re in rs_result.random_effects.items():
    rows.append({'participant_id': pid,
                 'intercept': fe_int + re['Group'],
                 'slope': fe_slope + re['reinforcement_rate']})
participant_params = pd.DataFrame(rows).sort_values('participant_id').reset_index(drop=True)
print(participant_params)

## Task 5: Compare Models Using AIC and BIC

Compare the three models (OLS, random-intercept, random-intercept-and-slope) using information criteria. Lower values indicate a better trade-off between fit and complexity.

**Note:** statsmodels does not populate `.aic`/`.bic` on `MixedLMResults` (they return `nan`), so we compute them from the log-likelihood. All models are fit by maximum likelihood here so the values are comparable.

Create a table summarizing the AIC and BIC for each model. Which model is preferred?

In [ ]:
def info_criteria(result, n):
    k = len(result.params) + 1  # +1 for the residual variance (scale)
    aic = -2 * result.llf + 2 * k
    bic = -2 * result.llf + k * np.log(n)
    return aic, bic

n = len(df)
ri_aic, ri_bic = info_criteria(ri_result, n)
rs_aic, rs_bic = info_criteria(rs_result, n)

comparison = pd.DataFrame({
    'model': ['OLS (pooled)', 'Random intercept', 'Random intercept + slope'],
    'AIC': [ols_result.aic, ri_aic, rs_aic],
    'BIC': [ols_result.bic, ri_bic, rs_bic],
}).round(2)
print(comparison)
print('\nPreferred by AIC:', comparison.loc[comparison['AIC'].idxmin(), 'model'])
print('Preferred by BIC:', comparison.loc[comparison['BIC'].idxmin(), 'model'])

**Question:** What do the AIC/BIC values tell you about the importance of accounting for individual differences in this dataset? Write your answer below.

Both criteria fall sharply as participant-level structure is added (AIC 1055 -> 821 -> 320; BIC 1061 -> 833 -> 338). The random-intercept-and-slope model is decisively preferred even under BIC's heavier penalty for extra parameters. Individual differences in *both* baseline responding and sensitivity to reinforcement are large, and a model that ignores them fits the data poorly.

## Task 6: Interpret Fixed and Random Effects

Using the random-intercept-and-slope model:

1. State in plain language what the fixed-effect intercept and slope represent.
2. State what the random-effect variances represent.
3. Which participants have the steepest and shallowest slopes? What might this mean behaviorally (e.g., sensitivity to reinforcement rate)?
4. Is there a correlation between participant intercepts and slopes? If so, what does it mean?

In [ ]:
steep = participant_params.loc[participant_params['slope'].idxmax()]
shallow = participant_params.loc[participant_params['slope'].idxmin()]
print(f"Steepest slope:  P{int(steep['participant_id'])} ({steep['slope']:.3f})")
print(f"Shallowest slope: P{int(shallow['participant_id'])} ({shallow['slope']:.3f})")

r, p = stats.pearsonr(participant_params['intercept'], participant_params['slope'])
print(f"\nIntercept-slope correlation: r = {r:.3f}, p = {p:.3f}")

The fixed-effect intercept (~8.4) is the average response rate predicted at zero reinforcement, and the fixed slope (~2.43) is the average increase in responses per minute for each additional reinforcer per minute. The random-effect variances describe how much participants deviate from those averages -- a large intercept variance means participants differ in baseline responding, and a non-zero slope variance means they differ in sensitivity to reinforcement. Here P8 has the steepest slope (~3.79, most sensitive) and P5 the shallowest (~0.99, nearly insensitive). The intercept-slope correlation is strongly negative (r = -0.97): participants with high baseline response rates tend to have shallower slopes, while low-baseline participants increase more steeply with reinforcement.

## Task 7: Visualize Participant-Level Regression Lines

Create a figure with:
1. A scatter plot of the raw data, colored by participant.
2. The OLS regression line (single line for all data) in black.
3. The participant-specific regression lines from the random-intercept-and-slope model, each in a different color.

This visualization should make it visually clear why a single regression line is inadequate.

In [ ]:
fig, ax = plt.subplots()
x_grid = np.linspace(df['reinforcement_rate'].min(), df['reinforcement_rate'].max(), 50)

for _, row in participant_params.iterrows():
    pid = int(row['participant_id'])
    g = df[df['participant_id'] == pid]
    pts = ax.scatter(g['reinforcement_rate'], g['response_rate'], alpha=0.5)
    ax.plot(x_grid, row['intercept'] + row['slope'] * x_grid,
            color=pts.get_facecolor()[0], linewidth=1.5)

ax.plot(x_grid,
        ols_result.params['Intercept'] + ols_result.params['reinforcement_rate'] * x_grid,
        color='black', linewidth=3, linestyle='--', label='OLS (pooled)')
ax.set_xlabel('Reinforcement rate (per min)')
ax.set_ylabel('Response rate (per min)')
ax.set_title('Participant-specific regression lines vs. pooled OLS')
ax.legend()
plt.tight_layout()
plt.show()

## Task 8: Visualize Random Effects

Create two plots:
1. A caterpillar plot (forest plot) showing the random intercepts for each participant with confidence intervals.
2. A caterpillar plot showing the random slopes for each participant.

**Hint:** You can extract the random effects from `result_rs.random_effects` and approximate confidence intervals using the conditional standard errors.

In [ ]:
# Point estimates of the random effects (deviations from the fixed effect)
re_int = {pid: re['Group'] for pid, re in rs_result.random_effects.items()}
re_slope = {pid: re['reinforcement_rate'] for pid, re in rs_result.random_effects.items()}

# Approximate SE bands from the estimated random-effect SDs (illustrative)
se_int = np.sqrt(rs_result.cov_re.loc['Group', 'Group'])
se_slope = np.sqrt(rs_result.cov_re.loc['reinforcement_rate', 'reinforcement_rate'])

def caterpillar(ax, effects, se, title):
    items = sorted(effects.items(), key=lambda kv: kv[1])
    labels = [f'P{pid}' for pid, _ in items]
    vals = [v for _, v in items]
    y = np.arange(len(vals))
    ax.errorbar(vals, y, xerr=1.96 * se, fmt='o', capsize=3)
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.set_title(title); ax.set_xlabel('Deviation from fixed effect')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
caterpillar(ax1, re_int, se_int, 'Random intercepts (~95% approx CI)')
caterpillar(ax2, re_slope, se_slope, 'Random slopes (~95% approx CI)')
plt.tight_layout()
plt.show()

## Task 9 (Optional): Time-Series Decomposition

Select one participant and treat their session-by-session `response_rate` as a short time series.

1. Plot the raw time series.
2. Use `statsmodels.tsa.seasonal.seasonal_decompose` (with `model='additive'` and an appropriate `period`) to decompose the series into trend, seasonal, and residual components.
3. Since our data may not have a true seasonal component over 20 sessions, focus on the trend and residual. What does the trend tell you about how this participant's behavior changed over sessions?

**Note:** With only 20 observations, this decomposition is illustrative rather than definitive. The goal is to introduce the concept of decomposing temporal data.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

p1 = df[df['participant_id'] == 1].sort_values('session').set_index('session')['response_rate']
decomp = seasonal_decompose(p1, model='additive', period=4)

fig = decomp.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.show()

print('Trend (start -> end):',
      round(decomp.trend.dropna().iloc[0], 2), '->', round(decomp.trend.dropna().iloc[-1], 2))

## Reflection

In a few sentences, address the following:

1. Why is multilevel modeling particularly important for behavioral research where data are collected across multiple participants and sessions?
2. How would you explain the difference between fixed effects and random effects to a colleague who has only used standard regression?
3. What are the limitations of the models you fit today (e.g., assumptions about the error structure, linearity)?

1. Behavioral data are almost always nested (sessions within participants, participants within groups), and pooling ignores that structure -- biasing standard errors and masking individual differences that are often the phenomenon of interest. 2. Fixed effects are the average relationships that apply to the whole population; random effects capture how individual units depart from those averages, with the model estimating the *distribution* of those departures rather than a separate free parameter per unit. 3. These models still assume linearity, normally distributed random effects and residuals, and homoscedastic within-participant errors; with only 8 participants and 20 sessions each the variance components (and the time-series decomposition) are estimated imprecisely.